In [12]:
# =========================
# IMPORTS
# =========================

import os
import re
import json
import math
import time
import pickle
import hashlib
import platform
import unicodedata
from dataclasses import dataclass
from collections import Counter, defaultdict
from typing import Dict, List, Tuple, Any, Optional

import numpy as np


In [ ]:
# =========================
# IMPORTS DESDE MÓDULO 
# =========================

import sys
sys.path.append(r"C:\Users\VLADIMIR\OneDrive\Documentos\Tesis UPEU\Código\src")

from bm25_index import (
    BM25Index,
    build_bm25_from_corpus,
    read_jsonl,
    sha256_file,
    tokenize_es,
    is_8_digits
)

In [13]:
# =========================
# CONFIGURACIÓN
# =========================

# (A) Rutas del proyecto 
BASE_DIR = r"C:\Users\VLADIMIR\OneDrive\Documentos\Tesis UPEU\Código"
DATA_PROCESSED_DIR = os.path.join(BASE_DIR, "data", "processed")

# (B) Archivo del corpus (mixto: NANDINA + Arancel/Notas/RGI se usa corpus curado para indexación)
CORPUS_FILE = "corpus_rag_v1_index.jsonl"
CORPUS_PATH = os.path.join(DATA_PROCESSED_DIR, CORPUS_FILE)

# (C) Directorio de salida para artefactos del índice
OUT_DIR = os.path.join(DATA_PROCESSED_DIR, "indexes")
os.makedirs(OUT_DIR, exist_ok=True)

BM25_OUT_PATH = os.path.join(OUT_DIR, "bm25_nandina8.pkl")
META_OUT_PATH = os.path.join(OUT_DIR, "bm25_nandina8_run_metadata.json")

# (D) Esquema del corpus (campos)
TYPE_FIELD = "tipo"
CODE_FIELD = "codigo"
TITLE_FIELD = "titulo"
TEXT_FIELD = "texto_index" # Campo limpio para indexación
FALLBACK_TEXT_FIELD = "texto" # fallback al texto original si por alguna razón texto_index viniera vacío


TARGET_TYPE = "nandina_8"  # Solo clases NANDINA-8

# (E) Parámetros BM25 (valores estándar en IR)
K1 = 1.5
B  = 0.75

# (F) Preprocesamiento
USE_STOPWORDS = True
STOPWORDS_ES = {
    "de","la","el","y","o","u","en","a","para","por","con","sin","del","al",
    "un","una","unos","unas","lo","las","los","su","sus","se","que","como",
    "mas","menos","muy","ya","no","si","es","son","ser","estar","esta","este",
    "estas","estos","entre","sobre","desde","hasta","segun","mediante",
    "tipo","producto","articulo","mercancia","codigo"
}

STOP = STOPWORDS_ES if USE_STOPWORDS else None

# (G) Control adicional (recomendado): validar que codigo sea exactamente 8 dígitos
ENFORCE_8_DIGITS = True

# (H) Consulta de prueba (sanity check)
TEST_QUERY = "camisetas de algodon para hombre"
TEST_TOP_N = 10

# Verificación básica de existencia del corpus
if not os.path.exists(CORPUS_PATH):
    raise FileNotFoundError(
        f"No se encontró el corpus en:\n{CORPUS_PATH}\n"
        f"Verifica BASE_DIR/DATA_PROCESSED_DIR/CORPUS_FILE."
    )

print("Corpus encontrado:", CORPUS_PATH)
print("Salida índice:", BM25_OUT_PATH)
print("Salida metadatos:", META_OUT_PATH)


Corpus encontrado: C:\Users\VLADIMIR\OneDrive\Documentos\Tesis UPEU\Código\data\processed\corpus_rag_v1_index.jsonl
Salida índice: C:\Users\VLADIMIR\OneDrive\Documentos\Tesis UPEU\Código\data\processed\indexes\bm25_nandina8.pkl
Salida metadatos: C:\Users\VLADIMIR\OneDrive\Documentos\Tesis UPEU\Código\data\processed\indexes\bm25_nandina8_run_metadata.json


In [16]:
# =========================
# CONSTRUCCIÓN + PERSISTENCIA + METADATOS
# =========================

t0 = time.time()

corpus_sha = sha256_file(CORPUS_PATH)
rows = read_jsonl(CORPUS_PATH)

bm25_index, stats = build_bm25_from_corpus(
    rows=rows,
    type_field=TYPE_FIELD,
    code_field=CODE_FIELD,
    title_field=TITLE_FIELD,
    text_field=TEXT_FIELD,
    target_type=TARGET_TYPE,
    k1=K1,
    b=B,
    stopwords=STOP,
    enforce_8_digits=ENFORCE_8_DIGITS
)

# Guardar índice
with open(BM25_OUT_PATH, "wb") as f:
    pickle.dump(bm25_index, f)

elapsed = time.time() - t0

# Metadatos (reproducibilidad)
run_metadata = {
    "notebook_name": "04_BM25_Indexacion_NANDINA.ipynb",
    "timestamp_unix": int(time.time()),
    "environment": {
        "python_version": platform.python_version(),
        "system": platform.system(),
        "release": platform.release(),
        "machine": platform.machine(),
    },
    "input": {
        "corpus_path": CORPUS_PATH,
        "corpus_sha256": corpus_sha,
        "schema": {
            "type_field": TYPE_FIELD,
            "code_field": CODE_FIELD,
            "title_field": TITLE_FIELD,
            "text_field": TEXT_FIELD,
            "fallback_text_field": FALLBACK_TEXT_FIELD if "FALLBACK_TEXT_FIELD" in globals() else None
        },

        "filter": {
            "target_type": TARGET_TYPE,
            "enforce_8_digits": ENFORCE_8_DIGITS
        }
    },
    "bm25_params": {
        "k1": K1,
        "b": B,
        "use_stopwords": USE_STOPWORDS,
        "stopwords_count": len(STOPWORDS_ES) if USE_STOPWORDS else 0
    },
    "index_stats": stats,
    "output": {
        "bm25_index_path": BM25_OUT_PATH,
        "metadata_path": META_OUT_PATH,
        "elapsed_seconds": float(elapsed)
    }
}

with open(META_OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(run_metadata, f, ensure_ascii=False, indent=2)

print("OK: Índice BM25 construido para NANDINA-8.")
print(" - Docs indexados:", stats["docs_indexed"])
print(" - Vocabulario:", stats["vocab_size"])
print(" - Corpus SHA-256:", corpus_sha)
print(" - Artefacto:", BM25_OUT_PATH)
print(" - Metadatos:", META_OUT_PATH)
print(" - Tiempo (s):", round(elapsed, 2))


OK: Índice BM25 construido para NANDINA-8.
 - Docs indexados: 7644
 - Vocabulario: 5646
 - Corpus SHA-256: 83768faae816b9d9b33a8fd36b73068d8b5f0b7a186e1c0f5b1c2c27580290f0
 - Artefacto: C:\Users\VLADIMIR\OneDrive\Documentos\Tesis UPEU\Código\data\processed\indexes\bm25_nandina8.pkl
 - Metadatos: C:\Users\VLADIMIR\OneDrive\Documentos\Tesis UPEU\Código\data\processed\indexes\bm25_nandina8_run_metadata.json
 - Tiempo (s): 0.16


In [18]:
# =========================
# CELDA 7: PRUEBA RÁPIDA (SANITY CHECK)
# =========================

# Cargar desde disco para verificar reutilización
with open(BM25_OUT_PATH, "rb") as f:
    bm25_loaded: BM25Index = pickle.load(f)

TEST_QUERY = "Computadora portátil con procesador Intel Core i5, memoria RAM 8 GB, disco sólido SSD 512 GB, pantalla LED de 14 pulgadas."
results = bm25_loaded.score(TEST_QUERY, top_n=TEST_TOP_N, stopwords=STOP)

print(f"Query de prueba: {TEST_QUERY}")
print(f"Top-{TEST_TOP_N} resultados:\n")

for rank, (doc_idx, score) in enumerate(results, start=1):
    codigo = bm25_loaded.doc_ids[doc_idx]
    texto  = bm25_loaded.doc_texts[doc_idx]
    print(f"{rank:02d}. NANDINA={codigo} | score={score:.4f} | {texto[:140]}...")


Query de prueba: Computadora portátil con procesador Intel Core i5, memoria RAM 8 GB, disco sólido SSD 512 GB, pantalla LED de 14 pulgadas.
Top-10 resultados:

01. NANDINA=28151100 | score=15.4455 | Sólido Sólido....
02. NANDINA=84717000 | score=13.5422 | Unidades de memoria Unidades de memoria....
03. NANDINA=85414100 | score=9.4663 | Diodos emisores de luz (LED) Diodos emisores de luz (LED)....
04. NANDINA=85395100 | score=8.6147 | Módulos de diodos emisores de luz (LED) Módulos de diodos emisores de luz (LED)....
05. NANDINA=85395200 | score=7.9036 | Lámparas y tubos de diodos emisores de luz (LED) Lámparas y tubos de diodos emisores de luz (LED)....
06. NANDINA=85411000 | score=7.3010 | Diodos, excepto los fotodiodos y los diodos emisores de luz (LED) Diodos, excepto los fotodiodos y los diodos emisores de luz (LED)....
